# product-taxonomy-bench: reproducible baselines

This notebook loads an anonymised snapshot directly from Hugging Face and runs:

- Stepwise p-adic linear regression (UMLLR-style)
- Decision tree (bag-of-tags)
- Small MLP (bag-of-tags)
- Zubarev simulated annealing p-adic regression

The p-adic baselines are implemented inline (pure Python) so the notebook runs
without installing any project-specific package.

Expected dataset layout in the HF repo:

- `paper/snapshot.json`, `paper/tags.jsonl.gz`, `paper/products-*.jsonl.gz`
- `latest/snapshot.json`, `latest/tags.jsonl.gz`, `latest/products-*.jsonl.gz`


In [ ]:
import json
import math
import os
import random
import re
import urllib.parse
import urllib.request
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Mapping, Sequence, Tuple

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

_PATH_SPLIT_RE = re.compile(r"[>/|]+")
_SEGMENT_NUMBER_RE = re.compile(r"-?\d+")


def parse_taxonomy_path(path: Any) -> Tuple[str, ...]:
    # Normalise a taxonomy path into a tuple of segments (root-to-leaf).

    if path is None:
        return ()

    if isinstance(path, str):
        stripped = path.strip()
        if not stripped:
            return ()
        parts = [part.strip() for part in _PATH_SPLIT_RE.split(stripped) if part.strip()]
        if parts:
            return tuple(parts)
        return (stripped,)

    if isinstance(path, Sequence) and not isinstance(path, (bytes, bytearray)):
        parts = [str(part).strip() for part in path if str(part).strip()]
        return tuple(parts)

    text = str(path).strip()
    return (text,) if text else ()


def parse_taxonomy_digits(path_value: str | None) -> Tuple[int, ...]:
    # Extract numeric digits from taxonomy path segments.

    if not path_value:
        return ()

    digits: List[int] = []
    for segment in parse_taxonomy_path(path_value):
        segment = segment.strip()
        if not segment:
            continue
        try:
            digits.append(int(segment))
            continue
        except ValueError:
            matches = _SEGMENT_NUMBER_RE.findall(segment)
            if matches:
                digits.extend(int(match) for match in matches)
                continue
    return tuple(digits)


def encode_path(digits: Sequence[int], base: int) -> int:
    value = 0
    for power, digit in enumerate(digits):
        value += digit * (base ** power)
    return value


def is_prime(value: int) -> bool:
    if value < 2:
        return False
    if value in {2, 3}:
        return True
    if value % 2 == 0:
        return False
    limit = int(math.isqrt(value)) + 1
    for factor in range(3, limit, 2):
        if value % factor == 0:
            return False
    return True


def next_prime(min_value: int) -> int:
    candidate = max(2, min_value + 1)
    while True:
        if is_prime(candidate):
            return candidate
        candidate += 1


def p_adic_distance(a: int, b: int, base: int) -> float:
    if a == b:
        return 0.0

    diff = abs(a - b)
    valuation = 0
    while diff % base == 0:
        diff //= base
        valuation += 1
    return base ** (-valuation)


@dataclass(frozen=True)
class ProductRecord:
    product_id: int
    tags: List[str]
    encoded_path: int
    cv_fold: int


@dataclass(frozen=True)
class BattleRecord:
    winner_tag: str
    loser_tag: str
    cv_fold: int | None


@dataclass(frozen=True)
class TagCoefficient:
    tag: str
    coefficient: int
    sequence: int


@dataclass(frozen=True)
class Prediction:
    product_id: int
    true_value: int
    predicted_value: int
    loss: float


@dataclass(frozen=True)
class UMLLRFoldResult:
    cv_fold: int
    coefficients: List[TagCoefficient]
    predictions: List[Prediction]
    loss: float
    default_prediction: int


def _tag_order(
    battles: Sequence[BattleRecord],
    holdout_fold: int,
    training_tags: Iterable[str],
) -> List[str]:
    wins: Dict[str, int] = {}
    losses: Dict[str, int] = {}

    for battle in battles:
        if battle.cv_fold == holdout_fold:
            continue
        wins[battle.winner_tag] = wins.get(battle.winner_tag, 0) + 1
        losses[battle.loser_tag] = losses.get(battle.loser_tag, 0) + 1

    ordered_tags = list({tag for tag in training_tags})
    for tag in ordered_tags:
        wins.setdefault(tag, 0)
        losses.setdefault(tag, 0)

    ordered_tags.sort(key=lambda tag: (-wins[tag], losses[tag], tag))
    return ordered_tags


def _select_coefficient(values: Sequence[int], base: int) -> int:
    unique_values = sorted(set(values))
    best_value = unique_values[0]
    best_loss = math.inf

    for candidate in unique_values:
        total_distance = sum(p_adic_distance(candidate, value, base) for value in values)
        if total_distance < best_loss or (
            math.isclose(total_distance, best_loss) and candidate < best_value
        ):
            best_loss = total_distance
            best_value = candidate

    return best_value


def _select_default_prediction(
    no_tag_values: Sequence[int],
    candidate_values: Sequence[int],
    base: int,
) -> int:
    if not no_tag_values:
        if candidate_values:
            return Counter(candidate_values).most_common(1)[0][0]
        return 0

    unique_candidates = sorted(set(candidate_values)) if candidate_values else [0]
    best_value = unique_candidates[0]
    best_loss = float("inf")

    for candidate in unique_candidates:
        total_loss = sum(p_adic_distance(candidate, value, base) for value in no_tag_values)
        if total_loss < best_loss or (total_loss == best_loss and candidate < best_value):
            best_loss = total_loss
            best_value = candidate

    return best_value


def umllr_run_fold(
    fold: int,
    records: Sequence[ProductRecord],
    battles: Sequence[BattleRecord],
    base: int,
) -> UMLLRFoldResult:
    training = [record for record in records if record.cv_fold != fold]
    testing = [record for record in records if record.cv_fold == fold]

    product_residuals: Dict[int, int] = {
        record.product_id: record.encoded_path for record in training
    }
    tag_to_products: Dict[str, List[int]] = {}
    for record in training:
        for tag in record.tags:
            tag_to_products.setdefault(tag, []).append(record.product_id)

    tag_order = _tag_order(battles, fold, tag_to_products.keys())

    coefficients: List[TagCoefficient] = []
    for sequence, tag in enumerate(tag_order):
        product_ids = tag_to_products.get(tag, [])
        values = [product_residuals[pid] for pid in product_ids]

        if values:
            coefficient = _select_coefficient(values, base)
            for pid in product_ids:
                product_residuals[pid] -= coefficient
        else:
            coefficient = 0

        coefficients.append(
            TagCoefficient(tag=tag, coefficient=coefficient, sequence=sequence)
        )

    coefficient_lookup = {entry.tag: entry.coefficient for entry in coefficients}

    no_tag_training_values = [
        record.encoded_path
        for record in training
        if sum(coefficient_lookup.get(tag, 0) for tag in record.tags) == 0
    ]
    all_training_values = [record.encoded_path for record in training]

    default_prediction = _select_default_prediction(
        no_tag_training_values, all_training_values, base
    )

    predictions: List[Prediction] = []
    total_loss = 0.0
    for record in testing:
        predicted = sum(coefficient_lookup.get(tag, 0) for tag in record.tags)
        if predicted == 0:
            predicted = default_prediction
        loss = p_adic_distance(predicted, record.encoded_path, base)
        total_loss += loss
        predictions.append(
            Prediction(
                product_id=record.product_id,
                true_value=record.encoded_path,
                predicted_value=predicted,
                loss=loss,
            )
        )

    return UMLLRFoldResult(
        cv_fold=fold,
        coefficients=coefficients,
        predictions=predictions,
        loss=total_loss,
        default_prediction=default_prediction,
    )


# Zubarev simulated annealing regression (p-adic)

@dataclass(frozen=True)
class ZubarevFoldResult:
    cv_fold: int
    coefficients: List[TagCoefficient]
    predictions: List[Prediction]
    loss: float
    default_prediction: int
    iterations_used: int


def _binomial(n: int, k: int) -> int:
    if k < 0:
        return 0
    if k == 0:
        return 1
    if k > abs(n) and n >= 0:
        return 0

    result = 1
    for i in range(k):
        result = result * (n - i) // (i + 1)
    return result


def _mahler_predict(s: int, weights: Sequence[int]) -> int:
    result = 0
    for k, w in enumerate(weights):
        result += w * _binomial(s, k)
    return result


def _compute_loss(
    records: Sequence[ProductRecord],
    coefficients: Mapping[str, int],
    mahler_weights: Sequence[int],
    default_prediction: int,
    base: int,
) -> float:
    total_loss = 0.0

    for record in records:
        s = sum(coefficients.get(tag, 0) for tag in record.tags)
        predicted = _mahler_predict(s, mahler_weights) if mahler_weights else s

        if predicted == 0 and not any(coefficients.get(tag, 0) != 0 for tag in record.tags):
            predicted = default_prediction

        total_loss += p_adic_distance(predicted, record.encoded_path, base)

    return total_loss


def _initialize_coefficients_umllr_style(
    training: Sequence[ProductRecord],
    battles: Sequence[BattleRecord],
    holdout_fold: int,
    base: int,
) -> Dict[str, int]:
    tag_to_products: Dict[str, List[int]] = {}
    for record in training:
        for tag in record.tags:
            tag_to_products.setdefault(tag, []).append(record.product_id)

    tag_order = _tag_order(battles, holdout_fold, set(tag_to_products.keys()))

    product_residuals: Dict[int, int] = {
        record.product_id: record.encoded_path for record in training
    }
    coefficients: Dict[str, int] = {}

    for tag in tag_order:
        product_ids = tag_to_products.get(tag, [])
        values = [product_residuals[pid] for pid in product_ids]

        if values:
            coefficient = _select_coefficient(values, base)
            coefficients[tag] = coefficient
            for pid in product_ids:
                product_residuals[pid] -= coefficient
        else:
            coefficients[tag] = 0

    return coefficients


def _stochastic_optimize(
    training: Sequence[ProductRecord],
    validation: Sequence[ProductRecord],
    initial_coefficients: Dict[str, int],
    base: int,
    *,
    mahler_degree: int = 0,
    max_iterations: int = 10000,
    initial_temperature: float = 1.0,
    cooling_rate: float = 0.9995,
    min_temperature: float = 0.001,
    perturbation_scale: int = 1000,
    seed: int | None = None,
) -> Tuple[Dict[str, int], List[int], float, int]:
    if seed is not None:
        random.seed(seed)

    coefficients = dict(initial_coefficients)
    tags = list(coefficients.keys())

    mahler_weights = [0] + [1] + [0] * (mahler_degree - 1) if mahler_degree > 0 else []

    all_values = sorted({r.encoded_path for r in training})
    default_prediction = all_values[0] if all_values else 0

    current_loss = _compute_loss(training, coefficients, mahler_weights, default_prediction, base)
    best_coefficients = dict(coefficients)
    best_mahler = list(mahler_weights)
    best_loss = current_loss

    temperature = initial_temperature
    iteration = 0

    while iteration < max_iterations and temperature > min_temperature:
        if not tags:
            break

        tag = random.choice(tags)
        old_value = coefficients.get(tag, 0)

        if random.random() < 0.5:
            power = random.randint(0, 5)
            sign = random.choice([-1, 1])
            delta = sign * (base ** power)
        else:
            delta = random.randint(-perturbation_scale, perturbation_scale)

        coefficients[tag] = old_value + delta
        new_loss = _compute_loss(training, coefficients, mahler_weights, default_prediction, base)

        accepted = False
        if new_loss < current_loss:
            current_loss = new_loss
            accepted = True
            if new_loss < best_loss:
                best_loss = new_loss
                best_coefficients = dict(coefficients)
                best_mahler = list(mahler_weights)
        elif temperature > 0:
            delta_loss = new_loss - current_loss
            try:
                acceptance_prob = math.exp(-delta_loss / temperature)
            except OverflowError:
                acceptance_prob = 0.0
            if random.random() < acceptance_prob:
                current_loss = new_loss
                accepted = True

        if not accepted:
            coefficients[tag] = old_value

        temperature *= cooling_rate
        iteration += 1

    return best_coefficients, best_mahler, best_loss, iteration


def zubarev_run_fold(
    fold: int,
    records: Sequence[ProductRecord],
    battles: Sequence[BattleRecord],
    base: int,
    *,
    mahler_degree: int = 0,
    max_iterations: int = 10000,
    seed: int | None = None,
    validation_fraction: float = 0.2,
    initialization_method: str = "umllr",
) -> ZubarevFoldResult:
    all_training = [r for r in records if r.cv_fold != fold]
    testing = [r for r in records if r.cv_fold == fold]

    fold_seed = seed + fold if seed is not None else None
    if fold_seed is not None:
        random.seed(fold_seed)

    shuffled = list(all_training)
    random.shuffle(shuffled)
    val_size = int(len(shuffled) * validation_fraction)
    validation = shuffled[:val_size]
    training = shuffled[val_size:]

    if initialization_method == "umllr":
        initial_coefficients = _initialize_coefficients_umllr_style(all_training, battles, fold, base)
    elif initialization_method == "zeros":
        all_tags: set[str] = set()
        for record in all_training:
            all_tags.update(record.tags)
        initial_coefficients = {tag: 0 for tag in all_tags}
    else:
        raise ValueError(f"Unknown initialization_method: {initialization_method}")

    optimized_coefficients, mahler_weights, train_loss, iterations_used = _stochastic_optimize(
        training,
        validation,
        initial_coefficients,
        base,
        mahler_degree=mahler_degree,
        max_iterations=max_iterations,
        seed=fold_seed,
    )

    all_training_values = [r.encoded_path for r in training]
    no_tag_training_values = [
        r.encoded_path
        for r in training
        if sum(optimized_coefficients.get(tag, 0) for tag in r.tags) == 0
    ]
    default_prediction = _select_default_prediction(no_tag_training_values, all_training_values, base)

    predictions: List[Prediction] = []
    total_loss = 0.0

    for record in testing:
        s = sum(optimized_coefficients.get(tag, 0) for tag in record.tags)
        predicted = _mahler_predict(s, mahler_weights) if mahler_weights else s
        if predicted == 0 and not any(optimized_coefficients.get(tag, 0) != 0 for tag in record.tags):
            predicted = default_prediction

        loss = p_adic_distance(predicted, record.encoded_path, base)
        total_loss += loss
        predictions.append(
            Prediction(
                product_id=record.product_id,
                true_value=record.encoded_path,
                predicted_value=predicted,
                loss=loss,
            )
        )

    ordered_tags = _tag_order(battles, fold, set(optimized_coefficients.keys()))
    coefficients = [
        TagCoefficient(tag=tag, coefficient=int(optimized_coefficients.get(tag, 0)), sequence=i)
        for i, tag in enumerate(ordered_tags)
    ]

    return ZubarevFoldResult(
        cv_fold=fold,
        coefficients=coefficients,
        predictions=predictions,
        loss=total_loss,
        default_prediction=default_prediction,
        iterations_used=iterations_used,
    )


In [ ]:
DATASET_ID = os.getenv("PRODUCT_TAXONOMY_BENCH_DATASET_ID", "gregb/product-taxonomy-bench")
REVISION = os.getenv("PRODUCT_TAXONOMY_BENCH_REVISION", "main")
SNAPSHOT = os.getenv("PRODUCT_TAXONOMY_BENCH_SNAPSHOT", "latest")
HF_TOKEN = os.getenv("HF_TOKEN")
MAX_PRODUCTS = int(os.getenv("PRODUCT_TAXONOMY_BENCH_MAX_PRODUCTS", "0")) or None


def _hf_headers() -> dict[str, str]:
    if HF_TOKEN:
        return {"Authorization": f"Bearer {HF_TOKEN}"}
    return {}


def hf_api_json(path: str) -> dict:
    url = f"https://huggingface.co/api/{path.lstrip('/')}"
    request = urllib.request.Request(url, headers=_hf_headers())
    with urllib.request.urlopen(request) as response:
        return json.load(response)


def hf_resolve_url(path: str) -> str:
    dataset_slug = DATASET_ID
    revision_slug = urllib.parse.quote(REVISION, safe="")
    path_slug = urllib.parse.quote(path.lstrip("/"), safe="/")
    return f"https://huggingface.co/datasets/{dataset_slug}/resolve/{revision_slug}/{path_slug}"


def load_json(path: str) -> dict:
    request = urllib.request.Request(hf_resolve_url(path), headers=_hf_headers())
    with urllib.request.urlopen(request) as response:
        return json.load(response)


def load_jsonl(path: str) -> pd.DataFrame:
    return pd.read_json(hf_resolve_url(path), lines=True, compression="infer")


snapshot_prefix = SNAPSHOT.strip("/") + "/"
dataset_meta = hf_api_json(f"datasets/{DATASET_ID}")
all_paths = [entry.get("rfilename", "") for entry in dataset_meta.get("siblings", [])]
snapshot_paths = [path for path in all_paths if path.startswith(snapshot_prefix)]
if not snapshot_paths:
    raise ValueError(
        f"Snapshot folder {SNAPSHOT!r} not found in dataset {DATASET_ID!r}. "
        f"Available top-level folders: {sorted({p.split('/', 1)[0] for p in all_paths if p})}"
    )

snapshot_json_path = snapshot_prefix + "snapshot.json"
if snapshot_json_path not in snapshot_paths:
    raise ValueError(f"Missing {snapshot_json_path!r} in dataset {DATASET_ID!r}")

if snapshot_prefix + "tags.jsonl.gz" in snapshot_paths:
    tags_path = snapshot_prefix + "tags.jsonl.gz"
elif snapshot_prefix + "tags.jsonl" in snapshot_paths:
    tags_path = snapshot_prefix + "tags.jsonl"
else:
    raise ValueError(f"Missing tags JSONL file in snapshot {SNAPSHOT!r}")

products_pattern = re.compile(
    rf"^{re.escape(snapshot_prefix)}products-\d+\.jsonl(?:\.gz)?$"
)
product_paths = sorted(path for path in snapshot_paths if products_pattern.match(path))
if not product_paths:
    raise ValueError(f"No products JSONL shards found for snapshot {SNAPSHOT!r}")

snapshot_metadata = load_json(snapshot_json_path)
tags = load_jsonl(tags_path).sort_values("tag_rank").reset_index(drop=True)
product_frames = [load_jsonl(path) for path in product_paths]
products_raw = pd.concat(product_frames, ignore_index=True)

if MAX_PRODUCTS:
    products_raw = (
        products_raw.sort_values("product_id_hash").head(MAX_PRODUCTS).reset_index(drop=True)
    )

products = products_raw[
    [
        "product_id_hash",
        "taxonomy_id",
        "taxonomy_path",
        "taxonomy_name",
        "cv_fold",
        "tag_count",
        "title_part_count",
    ]
].copy()

product_tags = (
    products_raw[["product_id_hash", "tag_features"]]
    .explode("tag_features", ignore_index=True)
    .dropna(subset=["tag_features"])
)
if not product_tags.empty:
    expanded = pd.json_normalize(product_tags["tag_features"])
    product_tags = pd.concat(
        [product_tags.drop(columns=["tag_features"]), expanded],
        axis=1,
    )
else:
    product_tags = pd.DataFrame(
        columns=["product_id_hash", "tag_id", "in_title", "title_part", "title_position"]
    )

product_tags = product_tags.dropna(subset=["tag_id"]).copy()
product_tags["tag_id"] = product_tags["tag_id"].astype(str)
product_tags["in_title"] = product_tags["in_title"].fillna(False).astype(bool)
product_tags["title_part"] = pd.to_numeric(product_tags["title_part"], errors="coerce")
product_tags["title_position"] = pd.to_numeric(
    product_tags["title_position"], errors="coerce"
)

title_tags = product_tags[
    product_tags["in_title"] & product_tags["title_position"].notna()
][["product_id_hash", "title_part", "tag_id", "title_position"]].copy()
title_tags["title_part"] = title_tags["title_part"].fillna(0).astype(int)
title_tags["title_position"] = title_tags["title_position"].astype(int)

print("dataset", DATASET_ID, "revision", REVISION)
print("snapshot folder", SNAPSHOT)
print("snapshot name", snapshot_metadata.get("snapshot_name"))
print("products", len(products), "tags", len(tags), "shards", len(product_paths))
products.head()


In [ ]:
# Build bag-of-tags sparse matrix X and label vector y

products = products.dropna(subset=["cv_fold"]).copy()
products["cv_fold"] = products["cv_fold"].astype(int)
products = products.sort_values("product_id_hash").reset_index(drop=True)

tag_to_col = {tag_id: i for i, tag_id in enumerate(tags["tag_id"].tolist())}
product_to_row = {pid: i for i, pid in enumerate(products["product_id_hash"].tolist())}

rows = []
cols = []
data = []

for pid, tag_id in product_tags[["product_id_hash", "tag_id"]].itertuples(index=False, name=None):
    row_idx = product_to_row.get(pid)
    col_idx = tag_to_col.get(tag_id)
    if row_idx is None or col_idx is None:
        continue
    rows.append(row_idx)
    cols.append(col_idx)
    data.append(1.0)

X = sparse.csr_matrix(
    (data, (rows, cols)),
    shape=(len(products), len(tags)),
    dtype=np.float32,
)

y = products["taxonomy_id"].to_numpy(dtype=object)

print("products", X.shape[0])
print("tags", X.shape[1])
print("taxonomies", len(set(y)))

In [ ]:
# Build taxonomy encodings + p-adic base for evaluation

taxonomy_paths = (
    products[["taxonomy_id", "taxonomy_path"]]
    .drop_duplicates(subset=["taxonomy_id"])
    .itertuples(index=False, name=None)
)

taxonomy_digits = {}
max_digit = 0
for taxonomy_id, taxonomy_path in taxonomy_paths:
    digits = parse_taxonomy_digits(taxonomy_path)
    taxonomy_digits[taxonomy_id] = digits
    if digits:
        max_digit = max(max_digit, max(digits))

base = next_prime(max_digit)
taxonomy_encoded = {
    taxonomy_id: encode_path(digits, base)
    for taxonomy_id, digits in taxonomy_digits.items()
}


def mean_padic_loss(y_true, y_pred) -> float:
    losses = []
    for t, p in zip(y_true, y_pred):
        losses.append(p_adic_distance(taxonomy_encoded[t], taxonomy_encoded[p], base))
    return float(np.mean(losses)) if losses else 0.0


print("prime base", base)
print("max digit", max_digit)


In [ ]:
# Decision tree + unconstrained baselines (5-fold CV using precomputed cv_fold)

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

folds = sorted(products["cv_fold"].unique().tolist())
unn_hidden = int(os.getenv("PRODUCT_TAXONOMY_BENCH_UNN_HIDDEN", "12"))
unn_max_iter = int(os.getenv("PRODUCT_TAXONOMY_BENCH_UNN_MAX_ITER", "80"))
ulr_max_iter = int(os.getenv("PRODUCT_TAXONOMY_BENCH_ULR_MAX_ITER", "200"))


def tree_effective_params(model) -> float:
    n_classes = model.n_classes_
    if isinstance(n_classes, np.ndarray):
        n_classes = int(np.max(n_classes))
    else:
        n_classes = int(n_classes)
    return float(model.tree_.node_count * math.log2(max(n_classes, 2)))


def dense_model_params(model) -> float:
    return float(
        sum(arr.size for arr in model.coefs_) + sum(arr.size for arr in model.intercepts_)
    )


def l1_logistic_nonzero_params(model) -> float:
    coef_nonzero = int(np.count_nonzero(model.coef_))
    intercept_nonzero = int(np.count_nonzero(model.intercept_))
    return float(coef_nonzero + intercept_nonzero)


def eval_cv(make_model, param_counter):
    fold_losses = []
    fold_acc = []
    fold_params = []

    for fold in folds:
        train_mask = products["cv_fold"].to_numpy() != fold
        test_mask = products["cv_fold"].to_numpy() == fold

        model = make_model()
        model.fit(X[train_mask], y[train_mask])
        y_pred = model.predict(X[test_mask])

        fold_losses.append(mean_padic_loss(y[test_mask], y_pred))
        fold_acc.append(float(np.mean(y_pred == y[test_mask])))
        fold_params.append(float(param_counter(model)))

    return (
        float(np.mean(fold_losses)),
        fold_losses,
        float(np.mean(fold_acc)),
        fold_acc,
        float(np.mean(fold_params)),
        fold_params,
    )


dt_mean_loss, dt_losses, dt_mean_acc, dt_acc, dt_mean_params, dt_params = eval_cv(
    lambda: DecisionTreeClassifier(class_weight="balanced", random_state=42),
    param_counter=tree_effective_params,
)

unn_mean_loss, unn_losses, unn_mean_acc, unn_acc, unn_mean_params, unn_params = eval_cv(
    lambda: MLPClassifier(
        hidden_layer_sizes=(unn_hidden,),
        activation="relu",
        alpha=1e-4,
        batch_size=256,
        max_iter=unn_max_iter,
        random_state=42,
    ),
    param_counter=dense_model_params,
)

ulr_mean_loss, ulr_losses, ulr_mean_acc, ulr_acc, ulr_mean_params, ulr_params = eval_cv(
    lambda: LogisticRegression(
        penalty="l1",
        solver="saga",
        C=1.0,
        max_iter=ulr_max_iter,
        n_jobs=-1,
        multi_class="multinomial",
        random_state=42,
    ),
    param_counter=l1_logistic_nonzero_params,
)

dummy_mean_loss, dummy_losses, dummy_mean_acc, dummy_acc, dummy_mean_params, dummy_params = eval_cv(
    lambda: DummyClassifier(strategy="most_frequent"),
    param_counter=lambda _model: 1.0,
)

print("Decision tree mean p-adic loss", dt_mean_loss, "mean acc", dt_mean_acc, "mean params", dt_mean_params)
print("Unconstrained NN mean p-adic loss", unn_mean_loss, "mean acc", unn_mean_acc, "mean params", unn_mean_params)
print("Unconstrained LR mean p-adic loss", ulr_mean_loss, "mean acc", ulr_mean_acc, "mean params", ulr_mean_params)
print("Dummy mean p-adic loss", dummy_mean_loss, "mean acc", dummy_mean_acc, "mean params", dummy_mean_params)


In [ ]:
# Build UMLLR/Zubarev records + battles from the anonymised snapshot

tags_by_product = defaultdict(list)
for pid, tag_id in product_tags[["product_id_hash", "tag_id"]].itertuples(index=False, name=None):
    tags_by_product[pid].append(tag_id)

fold_by_product = dict(products[["product_id_hash", "cv_fold"]].itertuples(index=False, name=None))

records = []
for idx, (pid, taxonomy_id, cv_fold) in enumerate(
    products[["product_id_hash", "taxonomy_id", "cv_fold"]].itertuples(index=False, name=None)
):
    records.append(
        ProductRecord(
            product_id=idx,
            tags=tags_by_product.get(pid, []),
            encoded_path=taxonomy_encoded[taxonomy_id],
            cv_fold=int(cv_fold),
        )
    )


# Battles are derived from tags that overlap title text. Each title part becomes its own arena.
# ordered is ascending by title_position; later tags win against earlier ones.

battles = []
group_cols = ["product_id_hash", "title_part"]
for (pid, part), group in title_tags.groupby(group_cols):
    fold = fold_by_product.get(pid)
    if fold is None:
        continue
    ordered = group.sort_values("title_position")[["tag_id", "title_position"]].to_numpy()
    for i in range(len(ordered)):
        for j in range(i + 1, len(ordered)):
            loser = str(ordered[i][0])
            winner = str(ordered[j][0])
            if loser == winner:
                continue
            battles.append(
                BattleRecord(winner_tag=winner, loser_tag=loser, cv_fold=int(fold))
            )

print("records", len(records))
print("battles", len(battles))


In [ ]:
# Stepwise p-adic linear regression (UMLLR-style)

umllr_fold_losses = []
umllr_nonzero_params = []
for fold in folds:
    result = umllr_run_fold(fold, records, battles, base)
    testing_count = sum(1 for r in records if r.cv_fold == fold)
    mean_loss = (result.loss / testing_count) if testing_count else 0.0
    umllr_fold_losses.append(mean_loss)
    umllr_nonzero_params.append(sum(1 for c in result.coefficients if c.coefficient != 0))

print("UMLLR mean p-adic loss", float(np.mean(umllr_fold_losses)))
print("UMLLR nonzero params (per fold)", umllr_nonzero_params)


In [ ]:
# Zubarev simulated annealing p-adic regression
# NOTE: this can be slow; reduce max_iterations for quick checks.

zubarev_max_iterations = int(
    os.getenv("PRODUCT_TAXONOMY_BENCH_ZUBAREV_MAX_ITERATIONS", "2000")
)

zub_fold_losses = []
zub_nonzero_params = []
for fold in folds:
    result = zubarev_run_fold(
        fold,
        records,
        battles,
        base,
        mahler_degree=0,
        max_iterations=zubarev_max_iterations,
        seed=42,
        initialization_method="umllr",
    )
    testing_count = sum(1 for r in records if r.cv_fold == fold)
    mean_loss = (result.loss / testing_count) if testing_count else 0.0
    zub_fold_losses.append(mean_loss)
    zub_nonzero_params.append(sum(1 for c in result.coefficients if c.coefficient != 0))

print("Zubarev mean p-adic loss", float(np.mean(zub_fold_losses)))
print("Zubarev nonzero params (per fold)", zub_nonzero_params)


In [ ]:
# Final model comparison inputs + parsimoniousness metric

PARSIMONY_BASELINE_SLOPE = -0.1
PARSIMONY_BASELINE_INTERCEPT = -0.2
PARSIMONY_BASELINE_LABEL = (
    f"log10(loss) = {PARSIMONY_BASELINE_SLOPE:+.1f} * log10(params) {PARSIMONY_BASELINE_INTERCEPT:+.1f}"
)
EPS = 1e-12


def add_parsimony_columns(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame["log10_params"] = np.log10(frame["params"].clip(lower=1.0))
    frame["log10_loss"] = np.log10(frame["mean_padic_loss"].clip(lower=EPS))
    frame["baseline_log10_loss"] = (
        PARSIMONY_BASELINE_SLOPE * frame["log10_params"] + PARSIMONY_BASELINE_INTERCEPT
    )
    frame["parsimony_score"] = (
        frame["baseline_log10_loss"] - frame["log10_loss"]
    )
    frame["better_than_baseline"] = frame["parsimony_score"] > 0
    return frame


model_points = [
    {
        "model": "Unconstrained Logistic Regression with L1",
        "short_label": "ULR",
        "params": float(ulr_mean_params),
        "mean_padic_loss": float(ulr_mean_loss),
        "mean_accuracy": float(ulr_mean_acc),
        "color": "#8b5cf6",
        "marker": "D",
    },
    {
        "model": "Unconstrained Neural Network with L1",
        "short_label": "UNN",
        "params": float(unn_mean_params),
        "mean_padic_loss": float(unn_mean_loss),
        "mean_accuracy": float(unn_mean_acc),
        "color": "#ec4899",
        "marker": "p",
    },
    {
        "model": "Decision Tree",
        "short_label": "Decision Tree",
        "params": float(dt_mean_params),
        "mean_padic_loss": float(dt_mean_loss),
        "mean_accuracy": float(dt_mean_acc),
        "color": "#14b8a6",
        "marker": "h",
    },
    {
        "model": "Importance-Optimised $p$-adic Linear Regression",
        "short_label": "Importance-Optimised",
        "params": float(np.mean(umllr_nonzero_params)),
        "mean_padic_loss": float(np.mean(umllr_fold_losses)),
        "mean_accuracy": np.nan,
        "color": "#0b6ce3",
        "marker": "o",
    },
    {
        "model": "Dummy Baseline",
        "short_label": "Dummy",
        "params": float(dummy_mean_params),
        "mean_padic_loss": float(dummy_mean_loss),
        "mean_accuracy": float(dummy_mean_acc),
        "color": "#94a3b8",
        "marker": "X",
    },
]

model_results = add_parsimony_columns(pd.DataFrame(model_points))
current_snapshot_summary = {
    "snapshot": SNAPSHOT,
    "snapshot_name": snapshot_metadata.get("snapshot_name") or SNAPSHOT,
    "num_products": int(len(products)),
    "num_tags": int(len(tags)),
    "num_taxonomies": int(products["taxonomy_id"].nunique()),
}

model_results[["model", "params", "mean_padic_loss", "mean_accuracy"]].sort_values(
    "params"
).reset_index(drop=True)

In [ ]:
# Unconstrained Models: Complexity vs Performance (Log Scale)
# Matches the style of padjective.symmachus.org/assets/unconstrained_log_chart.png

import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)

for row in model_results.itertuples(index=False):
    scatter_kwargs = {
        "label": row.model,
        "color": row.color,
        "s": 150,
        "alpha": 0.8,
        "marker": row.marker,
    }
    if row.marker not in ["+", "x", ".", ","]:
        scatter_kwargs["edgecolors"] = "white"
        scatter_kwargs["linewidths"] = 2
    ax.scatter(row.params, row.mean_padic_loss, **scatter_kwargs)
    ax.annotate(
        row.model,
        (row.params, row.mean_padic_loss),
        textcoords="offset points",
        xytext=(10, 5),
        fontsize=10,
        fontweight="bold",
        color=row.color,
    )

all_points = list(
    zip(
        model_results["params"].to_numpy(),
        model_results["mean_padic_loss"].to_numpy(),
    )
)

if len(all_points) >= 2:
    log_params = np.log10(model_results["params"].to_numpy())
    log_losses = np.log10(model_results["mean_padic_loss"].to_numpy())
    result = scipy_stats.linregress(log_params, log_losses)

    x_range = np.linspace(min(log_params) - 0.3, max(log_params) + 0.3, 100)
    y_fit = result.slope * x_range + result.intercept
    ax.plot(
        10 ** x_range,
        10 ** y_fit,
        "-",
        color="#ef4444",
        linewidth=2,
        alpha=0.7,
        label=f"Fit (R²={result.rvalue**2:.3f})",
    )

ax.set_xlabel("Number of Parameters (non-zero)", fontsize=12, fontweight="bold")
ax.set_ylabel("P-adic Loss (lower is better)", fontsize=12, fontweight="bold")
ax.set_title(
    "Unconstrained Models: Complexity vs Performance (Log Scale)",
    fontsize=14,
    fontweight="bold",
    pad=15,
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, alpha=0.3, linestyle="--")
ax.legend(loc="upper right", frameon=True, shadow=True, fontsize=9)

plt.tight_layout()
plt.show()

if len(all_points) >= 2:
    print("Regression: log10(loss) = slope × log10(params) + intercept")
    print("Slope\tIntercept\tR²\tp-value\tn")
    print(
        f"{result.slope:.4f}\t{result.intercept:.4f}\t{result.rvalue**2:.4f}\t{result.pvalue:.4g}\t{len(all_points)}"
    )


In [ ]:
# Model complexity vs p-adic loss (parsimoniousness baseline)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)

for row in model_results.itertuples(index=False):
    scatter_kwargs = {
        "label": row.model,
        "color": row.color,
        "s": 150,
        "alpha": 0.8,
        "marker": row.marker,
    }
    if row.marker not in ["+", "x", ".", ","]:
        scatter_kwargs["edgecolors"] = "white"
        scatter_kwargs["linewidths"] = 2
    ax.scatter(row.params, row.mean_padic_loss, **scatter_kwargs)
    ax.annotate(
        row.model,
        (row.params, row.mean_padic_loss),
        textcoords="offset points",
        xytext=(10, 5),
        fontsize=10,
        fontweight="bold",
        color=row.color,
    )

x_range = np.linspace(
    model_results["log10_params"].min() - 0.3,
    model_results["log10_params"].max() + 0.3,
    200,
)
baseline_log10_loss = PARSIMONY_BASELINE_SLOPE * x_range + PARSIMONY_BASELINE_INTERCEPT
ax.plot(
    10 ** x_range,
    10 ** baseline_log10_loss,
    "-",
    color="#ef4444",
    linewidth=2.2,
    alpha=0.8,
    label="Parsimoniousness baseline",
)

ax.set_xlabel("Number of Parameters (non-zero)", fontsize=12, fontweight="bold")
ax.set_ylabel("P-adic Loss (lower is better)", fontsize=12, fontweight="bold")
ax.set_title(
    "Model Complexity vs Performance (Parsimoniousness Baseline)",
    fontsize=14,
    fontweight="bold",
    pad=15,
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(True, alpha=0.3, linestyle="--")
ax.legend(loc="lower left", frameon=True, shadow=True, fontsize=8)

plt.tight_layout()
plt.show()

print("Parsimoniousness baseline:", PARSIMONY_BASELINE_LABEL)

In [ ]:
# Parsimoniousness by model (current snapshot)

import matplotlib.pyplot as plt

bar_df = model_results.sort_values("parsimony_score", ascending=False).reset_index(drop=True)
bar_colors = ["#16a34a" if value > 0 else "#dc2626" for value in bar_df["parsimony_score"]]

fig, ax = plt.subplots(figsize=(7, 4), dpi=150)
ax.bar(bar_df["short_label"], bar_df["parsimony_score"], color=bar_colors)
ax.axhline(0, color="#334155", linewidth=1)
ax.set_ylabel("Parsimony score (baseline − observed log10 loss)")
ax.set_title("Parsimoniousness by model")
ax.tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

print("Parsimoniousness baseline:", PARSIMONY_BASELINE_LABEL)
print("Parsimony score = baseline log10(loss) − observed log10(loss). Positive means better.")

In [ ]:
# Historical parsimoniousness across published Hugging Face snapshots

import warnings
from sklearn.exceptions import ConvergenceWarning

HISTORY_SNAPSHOTS = [
    name.strip()
    for name in os.getenv(
        "PRODUCT_TAXONOMY_BENCH_HISTORY_SNAPSHOTS",
        "first1000,paper,latest",
    ).split(",")
    if name.strip()
]


def load_snapshot_bundle(snapshot_name: str, max_products: int | None = None) -> dict[str, Any]:
    snapshot_prefix_local = snapshot_name.strip("/") + "/"
    snapshot_paths_local = [path for path in all_paths if path.startswith(snapshot_prefix_local)]
    if not snapshot_paths_local:
        raise ValueError(f"Snapshot folder {snapshot_name!r} not found in dataset {DATASET_ID!r}")

    snapshot_json_path_local = snapshot_prefix_local + "snapshot.json"
    if snapshot_json_path_local not in snapshot_paths_local:
        raise ValueError(f"Missing {snapshot_json_path_local!r} in dataset {DATASET_ID!r}")

    if snapshot_prefix_local + "tags.jsonl.gz" in snapshot_paths_local:
        tags_path_local = snapshot_prefix_local + "tags.jsonl.gz"
    elif snapshot_prefix_local + "tags.jsonl" in snapshot_paths_local:
        tags_path_local = snapshot_prefix_local + "tags.jsonl"
    else:
        raise ValueError(f"Missing tags JSONL file in snapshot {snapshot_name!r}")

    products_pattern_local = re.compile(
        rf"^{re.escape(snapshot_prefix_local)}products-\d+\.jsonl(?:\.gz)?$"
    )
    product_paths_local = sorted(
        path for path in snapshot_paths_local if products_pattern_local.match(path)
    )
    if not product_paths_local:
        raise ValueError(f"No products JSONL shards found for snapshot {snapshot_name!r}")

    snapshot_metadata_local = load_json(snapshot_json_path_local)
    tags_local = load_jsonl(tags_path_local).sort_values("tag_rank").reset_index(drop=True)
    product_frames_local = [load_jsonl(path) for path in product_paths_local]
    products_raw_local = pd.concat(product_frames_local, ignore_index=True)

    if max_products:
        products_raw_local = (
            products_raw_local.sort_values("product_id_hash")
            .head(max_products)
            .reset_index(drop=True)
        )

    products_local = products_raw_local[
        [
            "product_id_hash",
            "taxonomy_id",
            "taxonomy_path",
            "taxonomy_name",
            "cv_fold",
            "tag_count",
            "title_part_count",
        ]
    ].copy()

    product_tags_local = (
        products_raw_local[["product_id_hash", "tag_features"]]
        .explode("tag_features", ignore_index=True)
        .dropna(subset=["tag_features"])
    )
    if not product_tags_local.empty:
        expanded_local = pd.json_normalize(product_tags_local["tag_features"])
        product_tags_local = pd.concat(
            [product_tags_local.drop(columns=["tag_features"]), expanded_local],
            axis=1,
        )
    else:
        product_tags_local = pd.DataFrame(
            columns=["product_id_hash", "tag_id", "in_title", "title_part", "title_position"]
        )

    product_tags_local = product_tags_local.dropna(subset=["tag_id"]).copy()
    product_tags_local["tag_id"] = product_tags_local["tag_id"].astype(str)
    product_tags_local["in_title"] = product_tags_local["in_title"].fillna(False).astype(bool)
    product_tags_local["title_part"] = pd.to_numeric(
        product_tags_local["title_part"], errors="coerce"
    )
    product_tags_local["title_position"] = pd.to_numeric(
        product_tags_local["title_position"], errors="coerce"
    )

    title_tags_local = product_tags_local[
        product_tags_local["in_title"] & product_tags_local["title_position"].notna()
    ][["product_id_hash", "title_part", "tag_id", "title_position"]].copy()
    title_tags_local["title_part"] = title_tags_local["title_part"].fillna(0).astype(int)
    title_tags_local["title_position"] = title_tags_local["title_position"].astype(int)

    return {
        "snapshot_name": snapshot_metadata_local.get("snapshot_name") or snapshot_name,
        "products": products_local,
        "tags": tags_local,
        "product_tags": product_tags_local,
        "title_tags": title_tags_local,
    }



def build_snapshot_problem(snapshot_name: str, max_products: int | None = None) -> dict[str, Any]:
    bundle = load_snapshot_bundle(snapshot_name, max_products=max_products)
    products_eval = bundle["products"].dropna(subset=["cv_fold"]).copy()
    products_eval["cv_fold"] = products_eval["cv_fold"].astype(int)
    products_eval = products_eval.sort_values("product_id_hash").reset_index(drop=True)

    tags_eval = bundle["tags"].copy()
    product_tags_eval = bundle["product_tags"].copy()
    title_tags_eval = bundle["title_tags"].copy()

    tag_to_col_eval = {tag_id: i for i, tag_id in enumerate(tags_eval["tag_id"].tolist())}
    product_to_row_eval = {
        pid: i for i, pid in enumerate(products_eval["product_id_hash"].tolist())
    }

    rows_eval = []
    cols_eval = []
    data_eval = []
    for pid, tag_id in product_tags_eval[["product_id_hash", "tag_id"]].itertuples(index=False, name=None):
        row_idx = product_to_row_eval.get(pid)
        col_idx = tag_to_col_eval.get(tag_id)
        if row_idx is None or col_idx is None:
            continue
        rows_eval.append(row_idx)
        cols_eval.append(col_idx)
        data_eval.append(1.0)

    X_eval = sparse.csr_matrix(
        (data_eval, (rows_eval, cols_eval)),
        shape=(len(products_eval), len(tags_eval)),
        dtype=np.float32,
    )
    y_eval = products_eval["taxonomy_id"].to_numpy(dtype=object)

    taxonomy_paths_eval = (
        products_eval[["taxonomy_id", "taxonomy_path"]]
        .drop_duplicates(subset=["taxonomy_id"])
        .itertuples(index=False, name=None)
    )
    taxonomy_digits_eval = {}
    max_digit_eval = 0
    for taxonomy_id, taxonomy_path in taxonomy_paths_eval:
        digits = parse_taxonomy_digits(taxonomy_path)
        taxonomy_digits_eval[taxonomy_id] = digits
        if digits:
            max_digit_eval = max(max_digit_eval, max(digits))

    base_eval = next_prime(max_digit_eval)
    taxonomy_encoded_eval = {
        taxonomy_id: encode_path(digits, base_eval)
        for taxonomy_id, digits in taxonomy_digits_eval.items()
    }

    def mean_padic_loss_eval(y_true, y_pred) -> float:
        losses = []
        for truth, pred in zip(y_true, y_pred):
            losses.append(
                p_adic_distance(taxonomy_encoded_eval[truth], taxonomy_encoded_eval[pred], base_eval)
            )
        return float(np.mean(losses)) if losses else 0.0

    folds_eval = sorted(products_eval["cv_fold"].unique().tolist())

    tags_by_product_eval = defaultdict(list)
    for pid, tag_id in product_tags_eval[["product_id_hash", "tag_id"]].itertuples(index=False, name=None):
        tags_by_product_eval[pid].append(tag_id)

    fold_by_product_eval = dict(
        products_eval[["product_id_hash", "cv_fold"]].itertuples(index=False, name=None)
    )

    records_eval = []
    for idx, (pid, taxonomy_id, cv_fold) in enumerate(
        products_eval[["product_id_hash", "taxonomy_id", "cv_fold"]].itertuples(index=False, name=None)
    ):
        records_eval.append(
            ProductRecord(
                product_id=idx,
                tags=tags_by_product_eval.get(pid, []),
                encoded_path=taxonomy_encoded_eval[taxonomy_id],
                cv_fold=int(cv_fold),
            )
        )

    battles_eval = []
    for (pid, part), group in title_tags_eval.groupby(["product_id_hash", "title_part"]):
        fold = fold_by_product_eval.get(pid)
        if fold is None:
            continue
        ordered = group.sort_values("title_position")[["tag_id", "title_position"]].to_numpy()
        for i in range(len(ordered)):
            for j in range(i + 1, len(ordered)):
                loser = str(ordered[i][0])
                winner = str(ordered[j][0])
                if loser == winner:
                    continue
                battles_eval.append(
                    BattleRecord(winner_tag=winner, loser_tag=loser, cv_fold=int(fold))
                )

    return {
        "snapshot": snapshot_name,
        "snapshot_name": bundle["snapshot_name"],
        "products": products_eval,
        "tags": tags_eval,
        "X": X_eval,
        "y": y_eval,
        "folds": folds_eval,
        "base": base_eval,
        "mean_padic_loss": mean_padic_loss_eval,
        "records": records_eval,
        "battles": battles_eval,
        "num_products": int(len(products_eval)),
        "num_tags": int(len(tags_eval)),
        "num_taxonomies": int(products_eval["taxonomy_id"].nunique()),
    }



def eval_cv_snapshot(problem: dict[str, Any], make_model, param_counter):
    fold_losses = []
    fold_acc = []
    fold_params = []
    fold_assignments = problem["products"]["cv_fold"].to_numpy()

    for fold in problem["folds"]:
        train_mask = fold_assignments != fold
        test_mask = fold_assignments == fold

        model = make_model()
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=ConvergenceWarning)
            model.fit(problem["X"][train_mask], problem["y"][train_mask])
        y_pred = model.predict(problem["X"][test_mask])

        fold_losses.append(problem["mean_padic_loss"](problem["y"][test_mask], y_pred))
        fold_acc.append(float(np.mean(y_pred == problem["y"][test_mask])))
        fold_params.append(float(param_counter(model)))

    return (
        float(np.mean(fold_losses)),
        fold_losses,
        float(np.mean(fold_acc)),
        fold_acc,
        float(np.mean(fold_params)),
        fold_params,
    )



def attach_snapshot_metadata(frame: pd.DataFrame, problem: dict[str, Any]) -> pd.DataFrame:
    frame = frame.copy()
    frame["snapshot"] = problem["snapshot"]
    frame["snapshot_name"] = problem["snapshot_name"]
    frame["num_products"] = int(problem["num_products"])
    frame["num_tags"] = int(problem["num_tags"])
    frame["num_taxonomies"] = int(problem["num_taxonomies"])
    return frame



def evaluate_snapshot(snapshot_name: str) -> pd.DataFrame:
    if snapshot_name == SNAPSHOT:
        current_problem = {
            "snapshot": SNAPSHOT,
            "snapshot_name": current_snapshot_summary["snapshot_name"],
            "num_products": current_snapshot_summary["num_products"],
            "num_tags": current_snapshot_summary["num_tags"],
            "num_taxonomies": current_snapshot_summary["num_taxonomies"],
        }
        return attach_snapshot_metadata(model_results, current_problem)

    problem = build_snapshot_problem(snapshot_name, max_products=MAX_PRODUCTS)

    dt_mean_loss_eval, _, dt_mean_acc_eval, _, dt_mean_params_eval, _ = eval_cv_snapshot(
        problem,
        lambda: DecisionTreeClassifier(class_weight="balanced", random_state=42),
        param_counter=tree_effective_params,
    )
    unn_mean_loss_eval, _, unn_mean_acc_eval, _, unn_mean_params_eval, _ = eval_cv_snapshot(
        problem,
        lambda: MLPClassifier(
            hidden_layer_sizes=(unn_hidden,),
            activation="relu",
            alpha=1e-4,
            batch_size=64,
            max_iter=unn_max_iter,
            random_state=42,
        ),
        param_counter=dense_model_params,
    )
    ulr_mean_loss_eval, _, ulr_mean_acc_eval, _, ulr_mean_params_eval, _ = eval_cv_snapshot(
        problem,
        lambda: LogisticRegression(
            penalty="l1",
            solver="saga",
            multi_class="auto",
            class_weight="balanced",
            max_iter=ulr_max_iter,
            random_state=42,
        ),
        param_counter=l1_logistic_nonzero_params,
    )
    dummy_mean_loss_eval, _, dummy_mean_acc_eval, _, dummy_mean_params_eval, _ = eval_cv_snapshot(
        problem,
        lambda: DummyClassifier(strategy="most_frequent"),
        param_counter=lambda _: 1.0,
    )

    umllr_fold_losses_eval = []
    umllr_nonzero_params_eval = []
    for fold in problem["folds"]:
        result = umllr_run_fold(fold, problem["records"], problem["battles"], problem["base"])
        testing_count = sum(1 for record in problem["records"] if record.cv_fold == fold)
        mean_loss = (result.loss / testing_count) if testing_count else 0.0
        umllr_fold_losses_eval.append(mean_loss)
        umllr_nonzero_params_eval.append(
            sum(1 for coeff in result.coefficients if coeff.coefficient != 0)
        )

    snapshot_results = pd.DataFrame(
        [
            {
                "model": "Unconstrained Logistic Regression with L1",
                "short_label": "ULR",
                "params": float(ulr_mean_params_eval),
                "mean_padic_loss": float(ulr_mean_loss_eval),
                "mean_accuracy": float(ulr_mean_acc_eval),
                "color": "#8b5cf6",
                "marker": "D",
            },
            {
                "model": "Unconstrained Neural Network with L1",
                "short_label": "UNN",
                "params": float(unn_mean_params_eval),
                "mean_padic_loss": float(unn_mean_loss_eval),
                "mean_accuracy": float(unn_mean_acc_eval),
                "color": "#ec4899",
                "marker": "p",
            },
            {
                "model": "Decision Tree",
                "short_label": "Decision Tree",
                "params": float(dt_mean_params_eval),
                "mean_padic_loss": float(dt_mean_loss_eval),
                "mean_accuracy": float(dt_mean_acc_eval),
                "color": "#14b8a6",
                "marker": "h",
            },
            {
                "model": "Importance-Optimised $p$-adic Linear Regression",
                "short_label": "Importance-Optimised",
                "params": float(np.mean(umllr_nonzero_params_eval)),
                "mean_padic_loss": float(np.mean(umllr_fold_losses_eval)),
                "mean_accuracy": np.nan,
                "color": "#0b6ce3",
                "marker": "o",
            },
            {
                "model": "Dummy Baseline",
                "short_label": "Dummy",
                "params": float(dummy_mean_params_eval),
                "mean_padic_loss": float(dummy_mean_loss_eval),
                "mean_accuracy": float(dummy_mean_acc_eval),
                "color": "#94a3b8",
                "marker": "X",
            },
        ]
    )
    snapshot_results = add_parsimony_columns(snapshot_results)
    return attach_snapshot_metadata(snapshot_results, problem)


history_frames = []
for snapshot_name in HISTORY_SNAPSHOTS:
    print(f"Evaluating snapshot: {snapshot_name}")
    history_frames.append(evaluate_snapshot(snapshot_name))

history_results = pd.concat(history_frames, ignore_index=True)
latest_order = (
    history_results.sort_values(["short_label", "num_products"])
    .groupby("short_label", as_index=False)
    .tail(1)
    .sort_values("parsimony_score", ascending=False)
)
ordered_labels = latest_order["short_label"].tolist()

fig, (ax_scatter, ax_violin) = plt.subplots(
    1,
    2,
    figsize=(14, 5.5),
    dpi=150,
    gridspec_kw={"width_ratios": [1.8, 1.0]},
)
rng = np.random.default_rng(42)

for label in ordered_labels:
    model_history = history_results[history_results["short_label"] == label].sort_values(
        "num_products"
    )
    color = model_history["color"].iloc[0]
    marker = model_history["marker"].iloc[0]

    ax_scatter.plot(
        model_history["num_products"],
        model_history["parsimony_score"],
        color=color,
        linewidth=1.8,
        alpha=0.45,
        label=f"{label} (n={len(model_history)})",
    )

    scatter_kwargs = {
        "color": color,
        "s": 70,
        "alpha": 0.65,
        "marker": marker,
    }
    if marker not in ["+", "x", ".", ","]:
        scatter_kwargs["edgecolors"] = "white"
        scatter_kwargs["linewidths"] = 1.4
    ax_scatter.scatter(
        model_history["num_products"],
        model_history["parsimony_score"],
        **scatter_kwargs,
    )

    latest_row = model_history.iloc[-1]
    latest_kwargs = {
        "color": color,
        "s": 170,
        "alpha": 0.95,
        "marker": marker,
    }
    if marker not in ["+", "x", ".", ","]:
        latest_kwargs["edgecolors"] = "#0f172a"
        latest_kwargs["linewidths"] = 1.6
    ax_scatter.scatter(
        [latest_row["num_products"]],
        [latest_row["parsimony_score"]],
        **latest_kwargs,
    )

ax_scatter.axhline(0, color="#334155", linewidth=1.2, alpha=0.9)
ax_scatter.set_xscale("log")
ax_scatter.set_xlabel("Products in dataset snapshot", fontsize=12, fontweight="bold")
ax_scatter.set_ylabel("Parsimony score", fontsize=12, fontweight="bold")
ax_scatter.set_title(
    "Parsimoniousness vs dataset growth",
    fontsize=14,
    fontweight="bold",
    pad=12,
)
ax_scatter.grid(True, alpha=0.25, linestyle="--")
ax_scatter.legend(loc="best", frameon=True, shadow=True, fontsize=8, ncol=2)

positions = np.arange(1, len(ordered_labels) + 1)
for idx, label in enumerate(ordered_labels, start=1):
    model_history = history_results[history_results["short_label"] == label].sort_values(
        "num_products"
    )
    scores = model_history["parsimony_score"].to_numpy(dtype=float)
    color = model_history["color"].iloc[0]
    marker = model_history["marker"].iloc[0]

    if len(scores) >= 2 and float(np.ptp(scores)) > 0:
        violin = ax_violin.violinplot(
            [scores],
            positions=[idx],
            vert=False,
            widths=0.72,
            showmeans=False,
            showmedians=True,
            showextrema=False,
        )
        for body in violin["bodies"]:
            body.set_facecolor(color)
            body.set_edgecolor(color)
            body.set_alpha(0.18)
        if "cmedians" in violin:
            violin["cmedians"].set_color(color)
            violin["cmedians"].set_linewidth(2)

    jitter = rng.uniform(-0.12, 0.12, size=len(scores))
    ax_violin.scatter(
        scores,
        np.full(len(scores), idx, dtype=float) + jitter,
        color=color,
        marker=marker,
        s=48,
        alpha=0.75,
    )

ax_violin.axvline(0, color="#334155", linewidth=1.2, alpha=0.9)
ax_violin.set_yticks(positions)
ax_violin.set_yticklabels(ordered_labels)
ax_violin.set_xlabel("Parsimony score", fontsize=12, fontweight="bold")
ax_violin.set_title(
    "Historical score distribution",
    fontsize=14,
    fontweight="bold",
    pad=12,
)
ax_violin.grid(True, axis="x", alpha=0.25, linestyle="--")

fig.suptitle(
    "Parsimony stability over published snapshots",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

history_summary_rows = []
for label in ordered_labels:
    model_history = history_results[history_results["short_label"] == label].sort_values(
        "num_products"
    )
    scores = model_history["parsimony_score"].to_numpy(dtype=float)
    latest_row = model_history.iloc[-1]
    history_summary_rows.append(
        {
            "model": latest_row["model"],
            "short_label": label,
            "n_snapshots": int(len(model_history)),
            "mean_score": float(np.mean(scores)),
            "std_score": float(np.std(scores, ddof=1)) if len(scores) > 1 else 0.0,
            "span_score": float(np.max(scores) - np.min(scores)) if len(scores) else 0.0,
            "latest_score": float(latest_row["parsimony_score"]),
            "latest_products": int(latest_row["num_products"]),
        }
    )

history_summary = pd.DataFrame(history_summary_rows)
print("Snapshots evaluated:", ", ".join(HISTORY_SNAPSHOTS))
print("Parsimoniousness baseline:", PARSIMONY_BASELINE_LABEL)
history_summary

In [ ]:
# Results table (final summary)

results_table = model_results[
    [
        "model",
        "params",
        "mean_padic_loss",
        "mean_accuracy",
        "log10_params",
        "log10_loss",
        "baseline_log10_loss",
        "parsimony_score",
        "better_than_baseline",
    ]
].copy()

results_table["params"] = results_table["params"].round(1)
results_table["mean_padic_loss"] = results_table["mean_padic_loss"].round(6)
results_table["mean_accuracy"] = results_table["mean_accuracy"].round(4)
results_table["log10_params"] = results_table["log10_params"].round(4)
results_table["log10_loss"] = results_table["log10_loss"].round(4)
results_table["baseline_log10_loss"] = results_table["baseline_log10_loss"].round(4)
results_table["parsimony_score"] = results_table["parsimony_score"].round(4)

print("Parsimoniousness baseline:", PARSIMONY_BASELINE_LABEL)
results_table.sort_values("parsimony_score", ascending=False).reset_index(drop=True)